# NeuralNav — Intent Classification: Baseline vs DistilBERT

Run this on **Kaggle** with GPU enabled: Settings -> Accelerator -> GPU T4 x2. Internet must be ON (Settings -> Internet) since this pulls the dataset from the Hugging Face Hub.

**Real data**: [Bitext Customer Service Tagged Dataset](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) — ~27 intents across 10 categories, ~26.8k labeled examples.

**Low-data regime by design**: training uses only `MAX_PER_CLASS_TRAIN = 25` examples/class (not the full dataset). With hundreds of examples/class both the TF-IDF baseline and DistilBERT hit 98-99% — Bitext's phrasing is templated, so it's not a hard problem and the two models look nearly identical. Training on very little data instead surfaces the actual ML-vs-DL story: DistilBERT's pretrained language understanding lets it generalize from far fewer labeled examples than a from-scratch TF-IDF+LogReg model. Increase `MAX_PER_CLASS_TRAIN` if you'd rather report higher absolute accuracy than a data-efficiency comparison.

Outputs at the end (download from the Kaggle Output tab into your local `models/`, `data/`, and report-assets folders):
- `baseline_intent.joblib`, `bert_intent/` — trained models
- `baseline_report.json`, `bert_report.json` — per-class comparison reports
- `kb.json` — auto-derived knowledge base (sourced from a larger, separate sample of the dataset's own `response` column so it stays varied even though training uses few examples) to replace the placeholder `data/kb.json`
- `intents_real.csv` — the exact low-data training set used, for reproducibility
- `figures/` — PNGs for your report: confusion matrices (baseline vs BERT), per-class F1 comparison, headline accuracy/F1 bar chart, BERT loss curve, and CSVs of the most-confused intent pairs for each model

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas joblib matplotlib seaborn

## Load the real dataset (Bitext, via Hugging Face Hub)

In [ ]:
import json
import pandas as pd
from datasets import load_dataset

raw = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df_full = raw["train"].to_pandas()
print(df_full.shape)
df_full.head()

In [ ]:
# Columns are: flags, instruction, category, intent, response
df = df_full.rename(columns={"instruction": "text"})[["text", "intent", "category", "response"]].dropna()
df["text"] = df["text"].str.strip()

print("Intents:", df["intent"].nunique())
print(df["intent"].value_counts())

In [ ]:
# Two different sample sizes on purpose:
#  - MAX_PER_CLASS_KB: larger pool used only to build a varied kb.json (notebook 02)
#  - MAX_PER_CLASS_TRAIN: small pool used to train both classifiers below
#
# Why so small for training: with 300 examples/class both the TF-IDF baseline
# and DistilBERT hit 98-99% — Bitext's phrasing is templated/formulaic, so it's
# not a hard classification problem and the two models look nearly identical.
# Dropping to a low-data regime (25/class) is where the real ML-vs-DL story
# shows up: classical TF-IDF degrades much faster than a model that already
# has language understanding from pretraining. Bump this back up if you want
# higher absolute accuracy instead of the data-efficiency comparison.
MAX_PER_CLASS_KB = 300
MAX_PER_CLASS_TRAIN = 25

df_kb_source = (
    df.groupby("intent", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), MAX_PER_CLASS_KB), random_state=42))
    .reset_index(drop=True)
)

df = (
    df.groupby("intent", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), MAX_PER_CLASS_TRAIN), random_state=42))
    .reset_index(drop=True)
)
df.to_csv("/kaggle/working/intents_real.csv", index=False)
print("KB source pool:", df_kb_source.shape, "| Training set:", df.shape)

## Auto-derive the knowledge base from the dataset's own responses

Picks a handful of representative (text, response) pairs per intent to seed
`kb.json` for the retrieval step in notebook 02 — replaces the hand-written
placeholder KB with answers grounded in real data.

In [ ]:
EXAMPLES_PER_INTENT_IN_KB = 3
kb = []
for intent, group in df_kb_source.groupby("intent"):
    sample = group.drop_duplicates(subset="response").head(EXAMPLES_PER_INTENT_IN_KB)
    for i, row in enumerate(sample.itertuples()):
        kb.append({
            "id": f"kb_{intent}_{i}",
            "intent": intent,
            "question": row.text,
            "answer": row.response,
        })

json.dump(kb, open("/kaggle/working/kb.json", "w"), indent=2)
print(f"Built kb.json with {len(kb)} entries across {df_kb_source['intent'].nunique()} intents")

## Part 1 — Classical ML baseline (TF-IDF + Logistic Regression)

In [ ]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["intent"], test_size=0.2, random_state=42, stratify=df["intent"]
)

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=2000)),
])
baseline.fit(X_train, y_train)

preds = baseline.predict(X_test)
baseline_report = classification_report(y_test, preds, output_dict=True, zero_division=0)
print(classification_report(y_test, preds, zero_division=0))

joblib.dump(baseline, "/kaggle/working/baseline_intent.joblib")
json.dump(baseline_report, open("/kaggle/working/baseline_report.json", "w"), indent=2)

## Part 2 — Fine-tune DistilBERT (uses the T4 GPU)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
import numpy as np
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

BASE_MODEL = "distilbert-base-uncased"

le = LabelEncoder()
df["label"] = le.fit_transform(df["intent"])

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["intent"])

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=48)

train_ds = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df[["text", "label"]]).map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=len(le.classes_))

# Hyperparameters tuned for the LOW-DATA regime (MAX_PER_CLASS_TRAIN=25,
# ~540 training rows). The original batch_size=32/epochs=4 combo was tuned
# for the full 8100-row dataset and gives only ~17 steps/epoch here — nowhere
# near enough to fit a freshly-initialized 27-way classifier head (you'll see
# training loss stuck near ln(27)≈3.3-6 and accuracy collapse below the
# baseline if you use those settings on this little data). Smaller batches +
# many more epochs gives enough gradient steps for the small dataset; still
# trains in well under a minute on a T4.
args = TrainingArguments(
    output_dir="/kaggle/working/_bert_ckpt",
    num_train_epochs=30,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds)
trainer.train()

In [ ]:
preds = trainer.predict(test_ds)
pred_labels = np.argmax(preds.predictions, axis=1)
y_true = le.inverse_transform(test_df["label"].to_numpy())
y_pred = le.inverse_transform(pred_labels)

bert_report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, zero_division=0))

import os
OUT_DIR = "/kaggle/working/bert_intent"
os.makedirs(OUT_DIR, exist_ok=True)
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
json.dump(le.classes_.tolist(), open(f"{OUT_DIR}/label_classes.json", "w"))
json.dump(bert_report, open("/kaggle/working/bert_report.json", "w"), indent=2)
print("Done. Download /kaggle/working/{baseline_intent.joblib, bert_intent/, baseline_report.json, bert_report.json, kb.json, intents_real.csv}")

## Comparison table for the report

In [ ]:
comp = pd.DataFrame({
    "baseline_f1": {k: v["f1-score"] for k, v in baseline_report.items() if isinstance(v, dict)},
    "bert_f1": {k: v["f1-score"] for k, v in bert_report.items() if isinstance(v, dict)},
})
comp["delta"] = comp["bert_f1"] - comp["baseline_f1"]
comp.sort_values("delta", ascending=False)

## Visualizations for the report

All figures are saved as PNGs under `/kaggle/working/figures/` — download
that folder from the Kaggle Output tab and drop the images straight into
your report.

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DIR = "/kaggle/working/figures"
os.makedirs(FIG_DIR, exist_ok=True)
sns.set_theme(style="whitegrid")
LABELS = sorted(df["intent"].unique())

In [ ]:
# Confusion matrices — baseline vs BERT, side by side.
# Recomputed fresh here (not reusing earlier `preds`/`y_test`/`y_true`/`y_pred`
# variables) so this cell gives correct results even if you re-run cells out
# of order — stale variables from an earlier run are a classic notebook footgun.
from sklearn.metrics import confusion_matrix

preds = baseline.predict(X_test)
y_test_fresh = y_test

bert_preds = trainer.predict(test_ds)
y_pred = le.inverse_transform(np.argmax(bert_preds.predictions, axis=1))
y_true = le.inverse_transform(test_df["label"].to_numpy())

cm_baseline = confusion_matrix(y_test_fresh, preds, labels=LABELS)
cm_bert = confusion_matrix(y_true, y_pred, labels=LABELS)

fig, axes = plt.subplots(1, 2, figsize=(26, 11))
for ax, cm, title in zip(axes, [cm_baseline, cm_bert], ["Baseline (TF-IDF + LogReg)", "Fine-tuned DistilBERT"]):
    sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=LABELS, yticklabels=LABELS, ax=ax, cbar=False)
    ax.set_title(f"Confusion Matrix — {title}\n(low-data regime: {MAX_PER_CLASS_TRAIN}/class)")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=90)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Per-class F1: baseline vs BERT
plot_df = comp.drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore").sort_values("bert_f1")

fig, ax = plt.subplots(figsize=(10, 9))
y_pos = range(len(plot_df))
ax.barh([p - 0.2 for p in y_pos], plot_df["baseline_f1"], height=0.4, label="Baseline (TF-IDF+LogReg)", color="#94a3b8")
ax.barh([p + 0.2 for p in y_pos], plot_df["bert_f1"], height=0.4, label="DistilBERT", color="#4f46e5")
ax.set_yticks(list(y_pos))
ax.set_yticklabels(plot_df.index)
ax.set_xlabel("F1 score")
ax.set_title(f"Per-class F1 — Baseline vs DistilBERT (low-data regime: {MAX_PER_CLASS_TRAIN} examples/class)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/f1_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Headline accuracy comparison (the "DL wins with little data" summary chart)
summary = pd.DataFrame({
    "model": ["Baseline (TF-IDF+LogReg)", "DistilBERT"],
    "accuracy": [baseline_report["accuracy"], bert_report["accuracy"]],
    "macro_f1": [baseline_report["macro avg"]["f1-score"], bert_report["macro avg"]["f1-score"]],
})

fig, ax = plt.subplots(figsize=(7, 5))
x = range(len(summary))
ax.bar([p - 0.18 for p in x], summary["accuracy"], width=0.36, label="Accuracy", color="#0ea5e9")
ax.bar([p + 0.18 for p in x], summary["macro_f1"], width=0.36, label="Macro F1", color="#a855f7")
ax.set_xticks(list(x))
ax.set_xticklabels(summary["model"])
ax.set_ylim(0, 1.05)
ax.set_title(f"Overall accuracy/F1 — trained on only {MAX_PER_CLASS_TRAIN} examples/class")
for i, row in summary.iterrows():
    ax.text(i - 0.18, row["accuracy"] + 0.01, f'{row["accuracy"]:.2f}', ha="center")
    ax.text(i + 0.18, row["macro_f1"] + 0.01, f'{row["macro_f1"]:.2f}', ha="center")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/headline_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
summary

In [ ]:
# DistilBERT training/validation loss curve
history = pd.DataFrame(trainer.state.log_history)
train_loss = history.dropna(subset=["loss"])[["epoch", "loss"]]
eval_loss = history.dropna(subset=["eval_loss"])[["epoch", "eval_loss"]]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_loss["epoch"], train_loss["loss"], marker="o", label="train loss")
ax.plot(eval_loss["epoch"], eval_loss["eval_loss"], marker="o", label="validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("DistilBERT fine-tuning loss curve")
ax.legend()
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/bert_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## Most-confusable intent pairs

Off-diagonal confusion matrix entries — which intents each model mixes up
most. Useful even when overall accuracy looks similar between models, since
it shows *which* errors disappear when moving from baseline to DistilBERT.

In [ ]:
def top_confusions(cm, labels, top_n=10):
    pairs = []
    for i, true_label in enumerate(labels):
        for j, pred_label in enumerate(labels):
            if i != j and cm[i, j] > 0:
                pairs.append({"true": true_label, "predicted": pred_label, "count": int(cm[i, j])})
    return pd.DataFrame(pairs).sort_values("count", ascending=False).head(top_n).reset_index(drop=True)

baseline_confusions = top_confusions(cm_baseline, LABELS)
bert_confusions = top_confusions(cm_bert, LABELS)

print("Baseline — most confused pairs:")
display(baseline_confusions)
print("\nDistilBERT — most confused pairs:")
display(bert_confusions)

baseline_confusions.to_csv(f"{FIG_DIR}/baseline_confusions.csv", index=False)
bert_confusions.to_csv(f"{FIG_DIR}/bert_confusions.csv", index=False)

## Push everything to a Hugging Face Hub repo

Uploads the trained models, reports, KB, training set, and all report
figures to one HF repo, so you don't have to manually download/re-upload
each artifact between notebooks (or hand them to the backend later).

Needs an HF token with **write** access:
- In Kaggle: Add-ons -> Secrets -> add a secret named `HF_TOKEN`, then
  attach it to this notebook (Add-ons -> Secrets -> toggle it on). The cell
  below reads it via `kaggle_secrets` automatically; falls back to the
  `HF_TOKEN` env var if you set it another way.
- Set `HF_REPO_ID` below to `<your-username>/neuralnav-intent-models` (or
  any repo name you want) — it's created automatically if it doesn't exist.

In [ ]:
!pip install -q huggingface_hub

HF_REPO_ID = "unixio/neuralnav-intent-models"  # change to your HF username/repo
HF_REPO_TYPE = "model"  # use "dataset" instead if you'd rather store this as a dataset repo

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    print("No HF_TOKEN found (Kaggle secret or env var) — skipping upload. "
          "Add one via Add-ons -> Secrets to enable this cell.")
else:
    from huggingface_hub import HfApi, create_repo

    api = HfApi(token=hf_token)
    create_repo(HF_REPO_ID, repo_type=HF_REPO_TYPE, token=hf_token, exist_ok=True)

    api.upload_folder(
        folder_path="/kaggle/working",
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        token=hf_token,
        allow_patterns=[
            "baseline_intent.joblib",
            "baseline_report.json",
            "bert_intent/*",
            "bert_report.json",
            "kb.json",
            "intents_real.csv",
            "figures/*",
        ],
        ignore_patterns=["_bert_ckpt/*"],
        commit_message="Upload trained models, reports, KB, and report figures from notebook 01",
    )
    print(f"Uploaded to https://huggingface.co/{HF_REPO_ID}")